# M101 through the WCC in every filter, at SNR = 500 in the brightest pixel

This notebook renders the centre of M101 (NGC 5457) as the Lazuli Wide-field Context Camera would
record it in each of its imaging filters, with the exposure time of each filter chosen so that the
**brightest detector pixel reaches a signal-to-noise ratio of 500**. Everything goes through
`wcc_sim`: the sky image is draped onto the focal plane with `SkyImageComponent`, exposed with
`WCCObservation`, and written as the standard SCI / SATMASK / CAT / CLEAN / COMP FITS product.

**Input image.** M101 is not a PHANGS-HST galaxy, so the sky image comes from the
[Legacy Survey](https://www.legacysurvey.org/) DR10 cutout service (BASS/MzLS at this declination:
g, r and z at 0.262"/px, calibrated in nanomaggies). Each band is converted to Jy per pixel and
given a celestial WCS, which is all `SkyImageComponent` needs. The WCC frame is only
161" x 108", so the picture is the galaxy's inner disk and nucleus, not the whole 29' galaxy.

**How the exposure time is set.** The expected count rate of every detector pixel is known before
any noise is added (`WCCObservation.rate_image`), so the brightest pixel is found there and the
exposure time follows from the per-pixel noise model used by the simulator,
$\mathrm{SNR} = S t / \sqrt{S t + B t + n_\mathrm{reads}\,\sigma_\mathrm{read}^2}$.
A pixel at SNR 500 holds at least 250 000 electrons, about fifteen times the IMX455 well, so the
exposure is split into `n_reads` frames, each kept below the well (notebook
[02](../tutorial/02_detector_saturation_reads.ipynb)).

**Read the results with these caveats.**
- One 16.9 mas WCC pixel of diffuse galaxy light is faint (roughly 9 magnitudes fainter than the
  same surface brightness per square arcsecond), so SNR 500 *per pixel* needs very long exposures;
  the table below gives them in hours. SNR per arcsec$^2$ or per resolution element would be a far
  cheaper requirement; change `SNR_TARGET` or bin the frames if that is what you meant.
- The input is a ground-based image: stars and HII regions in it carry ~1.3" seeing, so they
  appear in the WCC frame at that resolution, not at the WCC's 70 mas. No Gaia stars are added on
  top, which would double-count the stars already in the image.
- Only g, r and z exist for this field. The u, i and bb filters and the three H-alpha filters are
  rendered from the nearest broadband image by `wcc_sim`'s flat-spectrum transfer
  (`SOURCE_BAND` below). The H-alpha frames therefore show continuum only: the input has no
  emission-line map, so real HII regions would be much brighter than shown.

Needs `wcc_sim` with its image extra (`pip install 'wcc-sim[image]'` for `reproject`) and network
access to legacysurvey.org the first time it runs; the cutouts are cached in `OUT_DIR`.

In [ ]:
import gc
import urllib.request
from pathlib import Path

import numpy as np
import matplotlib.pyplot as plt
from astropy.io import fits
from astropy.table import Table
from astropy.visualization import simple_norm
from astropy.wcs import WCS

from wcc_sim import SkyImageComponent, WCCObservation
from wcc_sim.detectors import detector_frame, saturation_level_e
from wcc_sim.plotting import block_max
from wcc_sim.starflux import sky_and_dark_rates

## Parameters

Change these first; leave the calculation cells intact. `SHAPE = None` renders the full
9568 x 6380 frame (about 2 GB of arrays per filter while it runs, so the loop frees each frame
before the next); `SHAPE = (2048, 2048)` is a 35" quick look that runs in a fraction of the time.

In [ ]:
TARGET = "M101"
RA, DEC = 210.80243, 54.34875         # NGC 5457 nucleus (NED, J2000), degrees
SNR_TARGET = 500.0                    # required SNR in the brightest detector pixel
SEED = 2026

# Every in-focus WCC imaging filter on the IMX455. Drop the three H-alpha entries to save time:
# the input image has no emission-line map, so they only show continuum.
FILTERS = ["zwo:u", "zwo:g", "zwo:r", "zwo:i", "zwo:z", "zwo:bb",
           "zwo:halpha2", "zwo:halpha6", "zwo:halpha20"]

# Which Legacy Survey band feeds each WCC filter (only g, r, z exist north of Dec +32).
SOURCE_BAND = {"zwo:u": "g", "zwo:g": "g", "zwo:r": "r", "zwo:i": "r", "zwo:z": "z", "zwo:bb": "r",
               "zwo:halpha2": "r", "zwo:halpha6": "r", "zwo:halpha20": "r"}

SHAPE = None                          # None = full frame; (2048, 2048) for a quick look
LAYER = "ls-dr10"                     # Legacy Survey layer served by the cutout service
PIXSCALE = 0.262                      # arcsec per input pixel (native)
CUTOUT_ARCSEC = 240                   # cutout side; the WCC frame is 161" x 108" (194" diagonal)
LOCAL_IMAGES = {}                     # {"r": "my_m101_r.fits"} to use your own Jy/pix or MJy/sr image instead
OUT_DIR = Path("m101_wcc")
PREVIEW_BIN = 8                       # block-max factor for the preview panels
ZOOM_HALF = 20                        # half-width (px) of the brightest-pixel cutouts

## 1. The sky image: a Legacy Survey cutout in Jy per pixel

One request per band keeps the band bookkeeping unambiguous. Pixel values are nanomaggies
(1 nanomaggy = 3.631 x 10$^{-6}$ Jy); `SkyImageComponent` requires finite, non-negative pixels,
so the few noise pixels below zero and any uncovered (NaN) pixels are set to zero, which adds a
negligible positive bias to the faint outskirts. The header keeps the source band in `SRCBAND`
rather than `BAND`, because `BAND` is reserved for images already normalised in a WCC filter.

In [ ]:
NANOMAGGY_JY = 3.631e-6

def fetch_legacy_band(band, ra, dec, size_arcsec, pixscale, layer, out_dir):
    '''Download one band of the Legacy Survey cutout at (ra, dec); cached on disk.'''
    size = int(np.ceil(size_arcsec / pixscale))
    path = out_dir / f"legacy_{layer}_{band}_{size}px.fits"
    if not path.exists():
        url = (f"https://www.legacysurvey.org/viewer/fits-cutout?ra={ra:.6f}&dec={dec:.6f}"
               f"&layer={layer}&pixscale={pixscale}&size={size}&bands={band}")
        print("downloading", url)
        urllib.request.urlretrieve(url, path)
    return path

def to_jy_per_pixel(src_path, band, out_path):
    '''Nanomaggy cutout -> 2-D Jy/pix image with a celestial WCS (what SkyImageComponent reads).'''
    with fits.open(src_path) as hdul:
        data = np.squeeze(np.asarray(hdul[0].data, dtype=float))
        wcs = WCS(hdul[0].header).celestial
    if data.ndim != 2:
        raise ValueError(f"{src_path}: expected one image plane, got shape {data.shape}")
    n_nan = int(np.sum(~np.isfinite(data)))
    n_neg = int(np.sum(data < 0))
    data = np.clip(np.nan_to_num(data, nan=0.0), 0.0, None) * NANOMAGGY_JY
    hdr = wcs.to_header()
    hdr["BUNIT"] = "Jy/pix"
    hdr["SRCBAND"] = (band, "Legacy Survey band of the pixel values")
    hdr["SRCLAYER"] = (LAYER, "Legacy Survey layer")
    fits.PrimaryHDU(data.astype(np.float32), hdr).writeto(out_path, overwrite=True)
    print(f"{band}: {data.shape[1]} x {data.shape[0]} px, {n_nan} NaN and {n_neg} negative pixels set to 0, "
          f"total {data.sum():.4f} Jy, brightest input pixel {1e3 * data.max():.3f} mJy")
    return out_path

OUT_DIR.mkdir(exist_ok=True)
images = {}
for band in sorted({SOURCE_BAND[f] for f in FILTERS}):
    if band in LOCAL_IMAGES:
        images[band] = Path(LOCAL_IMAGES[band])
        print(f"{band}: using {images[band]}")
        continue
    raw = fetch_legacy_band(band, RA, DEC, CUTOUT_ARCSEC, PIXSCALE, LAYER, OUT_DIR)
    images[band] = to_jy_per_pixel(raw, band, OUT_DIR / f"{TARGET.lower()}_{band}_jy_per_pix.fits")

In [ ]:
fig, axes = plt.subplots(1, len(images), figsize=(5 * len(images), 5))
for ax, (band, path) in zip(np.atleast_1d(axes), images.items()):
    data = fits.getdata(path)
    ax.imshow(data, origin="lower", cmap="gray_r", norm=simple_norm(data, "asinh", percent=99.7))
    ax.set_title(f"input {band} [Jy/pix], {data.shape[1] * PIXSCALE:.0f}\" across")
    ax.set_axis_off()
plt.show()

## 2. Exposure time for SNR = 500 in one pixel

For a pixel receiving $S$ e$^-$/s of source light on a background $B$ e$^-$/s (zodiacal sky plus
dark current, a few hundredths of an electron per second here) in $n$ co-added reads with read
noise $\sigma_R$ per read, the simulator's noise model gives

$$\mathrm{SNR}(t) = \frac{S t}{\sqrt{S t + B t + n\,\sigma_R^2}} .$$

Setting SNR to the target gives a quadratic in $t$. The number of reads is then the smallest $n$
that keeps the per-frame expectation $(S + B)\,t / n$ below the per-frame electron ceiling (the
IMX455 well, about 16 000 e$^-$), with a 5% margin; since $n$ enters the read-noise term, the two
are iterated until they agree. The ceiling the simulator applies is `saturation_level_e`, the
smaller of the well depth and the ADC full scale.

In [ ]:
def solve_exposure(rate_peak, background, read_noise, ceiling_e, snr, min_frame_s=0.25, margin=0.95):
    '''(exptime [s], n_reads) giving `snr` in a pixel that receives `rate_peak` e-/s of source light.'''
    n = 1
    for _ in range(100):
        a = rate_peak ** 2
        b = -(snr ** 2) * (rate_peak + background)
        c = -(snr ** 2) * n * read_noise ** 2
        t = (-b + np.sqrt(b * b - 4.0 * a * c)) / (2.0 * a)
        n_new = max(1, int(np.ceil((rate_peak + background) * t / (margin * ceiling_e))))
        if n_new == n:
            break
        n = n_new
    if t / n < min_frame_s:
        raise ValueError(f"{t / n:.3f} s frames are shorter than the {min_frame_s} s minimum frame")
    return float(t), int(n)

def model_snr(source_e, total_e, n_reads, read_noise):
    '''Per-pixel SNR under the simulator's noise model: Poisson on the full expectation plus n reads.'''
    return source_e / np.sqrt(total_e + n_reads * read_noise ** 2)

## 3. Render every filter

Per filter: place the detector on the nucleus (`detector_frame`), drape the matching band
(`WCCObservation`), take the noiseless count-rate image, find its brightest pixel, solve for the
exposure, expose with that time and read split, and write the FITS product. The SNR of the
brightest pixel is then recomputed from the exposed frame's `image_clean` (the expectation the noise
was drawn from) as a check on the solver, and the standardised residual
$(\mathrm{SCI} - \mathrm{CLEAN}) / \sigma_\mathrm{model}$ over the frame should have a standard
deviation of 1 if the noise model used here is the one the simulator applied. Each frame is freed
before the next filter; only a block-max preview and a cutout around the brightest pixel are kept.

In [ ]:
rows, previews, zooms = [], {}, {}
for filt in FILTERS:
    band = SOURCE_BAND[filt]
    comp = SkyImageComponent.from_fits(str(images[band]), label=f"{TARGET} {band}")
    frame = detector_frame(filt, (RA, DEC), shape=SHAPE)
    obs = WCCObservation(comp, frame)

    rate = obs.rate_image()                                      # e-/s per detector pixel, noiseless
    iy, ix = np.unravel_index(int(np.argmax(rate)), rate.shape)
    peak = float(rate[iy, ix])
    sky, dark = sky_and_dark_rates(obs.sim)
    read_noise = float(obs.sim.sensor.read_noise.value)
    well = obs.sim.sensor.meta.get("well_depth")
    ceiling = min(saturation_level_e(obs.sim.sensor), float(well) if well is not None else np.inf)
    t, n = solve_exposure(peak, sky + dark, read_noise, ceiling, SNR_TARGET)

    field = obs.expose(exptime=t, n_reads=n, seed=SEED, add_noise=True)
    source_e = peak * t
    snr_check = model_snr(source_e, float(field.image_clean[iy, ix]), n, read_noise)
    sub = (slice(None, None, 4), slice(None, None, 4))             # a 1/16 sample of the frame for the residual check
    z = (field.image_e[sub] - field.image_clean[sub]) / np.sqrt(field.image_clean[sub] + n * read_noise ** 2)
    z_std = float(np.std(z[~field.saturation_mask[sub]]))
    ra_pk, dec_pk = (float(v) for v in field.wcs.pixel_to_world_values(ix, iy))

    path = OUT_DIR / f"{TARGET.lower()}_{filt.replace(':', '_')}_snr{SNR_TARGET:.0f}.fits"
    field.write(path)

    rows.append(dict(filter=filt, source_band=band, peak_e_s=peak, exptime_s=t, exptime_h=t / 3600.0,
                     n_reads=n, frame_s=t / n, peak_e=source_e, snr_check=snr_check,
                     saturated_px=int(field.saturation_mask.sum()), z_std=z_std,
                     peak_x=int(ix), peak_y=int(iy), peak_ra=ra_pk, peak_dec=dec_pk,
                     sky_e_s=sky, dark_e_s=dark, read_noise=read_noise, ceiling_e=ceiling, file=path.name))
    previews[filt] = block_max(field.image_e, PREVIEW_BIN)
    ny, nx = field.image_e.shape
    y0, y1 = max(iy - ZOOM_HALF, 0), min(iy + ZOOM_HALF + 1, ny)
    x0, x1 = max(ix - ZOOM_HALF, 0), min(ix + ZOOM_HALF + 1, nx)
    zooms[filt] = (field.image_e[y0:y1, x0:x1].copy(), ix - x0, iy - y0)
    print(f"{filt:13s} <- {band}: brightest pixel {peak:9.3f} e-/s at ({ix}, {iy}); "
          f"exposure {t:11.1f} s = {t / 3600:7.2f} h in {n:3d} reads of {t / n:8.1f} s; "
          f"model SNR {snr_check:6.1f}; {int(field.saturation_mask.sum())} saturated px; "
          f"residual std {z_std:.3f}; wrote {path.name}")

    del field, obs, comp, rate, z
    gc.collect()

## 4. Results

`exptime_h` is the total integration per filter and `frame_s` the length of each of the
`n_reads` frames. `snr_check` is the brightest pixel's SNR recomputed from the exposed frame and
should equal `SNR_TARGET`; `saturated_px` should be 0; `z_std` should be close to 1.

In [ ]:
results = Table(rows=rows)
for col, fmt in (("peak_e_s", "%.3f"), ("exptime_s", "%.0f"), ("exptime_h", "%.2f"), ("frame_s", "%.1f"),
                 ("peak_e", "%.0f"), ("snr_check", "%.1f"), ("z_std", "%.3f"),
                 ("peak_ra", "%.5f"), ("peak_dec", "%.5f"), ("sky_e_s", "%.4f"), ("dark_e_s", "%.4f"),
                 ("read_noise", "%.2f"), ("ceiling_e", "%.0f")):
    results[col].format = fmt
results.write(OUT_DIR / f"{TARGET.lower()}_snr{SNR_TARGET:.0f}_exposures.csv", overwrite=True)
results["filter", "source_band", "peak_e_s", "exptime_h", "n_reads", "frame_s", "snr_check", "saturated_px", "z_std",
        "peak_x", "peak_y", "peak_ra", "peak_dec"]

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4))
ax.bar(results["filter"], results["exptime_h"], color="#00798c")
ax.set(ylabel=f"exposure for SNR = {SNR_TARGET:.0f} in the brightest pixel [h]", yscale="log")
ax.tick_params(axis="x", rotation=45)
for f, h, n in zip(results["filter"], results["exptime_h"], results["n_reads"]):
    ax.text(f, h, f"{n} reads", ha="center", va="bottom", fontsize=8)
plt.tight_layout()
plt.show()

## 5. The frames

Electrons per full exposure, asinh stretch per panel (the exposures differ by filter, so a common
stretch would hide the fainter bands). The red circle marks the brightest pixel the exposure was
tuned on. Panels are block-max reduced by `PREVIEW_BIN` for display; the FITS files hold the full
frames.

In [ ]:
ncol = 3
nrow = int(np.ceil(len(FILTERS) / ncol))
fig, axes = plt.subplots(nrow, ncol, figsize=(6 * ncol, 4.3 * nrow), squeeze=False)
for ax, row in zip(axes.ravel(), rows):
    img = previews[row["filter"]]
    ax.imshow(img, origin="lower", cmap="gray_r", norm=simple_norm(img, "asinh", percent=99.7),
              interpolation="nearest")
    ax.scatter(row["peak_x"] / PREVIEW_BIN, row["peak_y"] / PREVIEW_BIN, s=200, facecolor="none", edgecolor="#d1495b", lw=1.2)
    ax.set_title(f"{row['filter']} (from {row['source_band']}): {row['exptime_h']:.1f} h, {row['n_reads']} reads")
    ax.set_axis_off()
for ax in axes.ravel()[len(rows):]:
    ax.set_axis_off()
fig.suptitle(f"{TARGET} centre, WCC frame, e- per exposure tuned to SNR = {SNR_TARGET:.0f} in the brightest pixel", y=1.0)
plt.tight_layout()
plt.show()

## 6. The brightest pixel

A `2 * ZOOM_HALF + 1` px cutout of the noisy frame around the pixel the exposure was tuned on
(cross-hair), with its sky position. If it sits on a foreground star rather than on the nucleus, the
coordinates in the results table say so; a star smeared by ground seeing has a higher surface
brightness than the galaxy and so can win the argmax.

In [ ]:
fig, axes = plt.subplots(nrow, ncol, figsize=(4.2 * ncol, 4.2 * nrow), squeeze=False)
for ax, row in zip(axes.ravel(), rows):
    cut, cx, cy = zooms[row["filter"]]
    im = ax.imshow(cut, origin="lower", cmap="magma", interpolation="nearest")
    ax.axhline(cy, color="w", lw=0.6, alpha=0.7)
    ax.axvline(cx, color="w", lw=0.6, alpha=0.7)
    ax.set_title(f"{row['filter']}: {row['peak_e']:.0f} e- expected\n"
                 f"RA {row['peak_ra']:.5f}, Dec {row['peak_dec']:+.5f}", fontsize=9)
    ax.set_xticks([]); ax.set_yticks([])
    plt.colorbar(im, ax=ax, fraction=0.046, label="e-")
for ax in axes.ravel()[len(rows):]:
    ax.set_axis_off()
plt.tight_layout()
plt.show()

## Summary

```python
comp  = SkyImageComponent.from_fits("m101_r_jy_per_pix.fits")          # any WCS + BUNIT image
obs   = WCCObservation(comp, detector_frame("zwo:r", (RA, DEC), shape=None))
rate  = obs.rate_image()                                                 # e-/s per pixel, noiseless
t, n  = solve_exposure(rate.max(), sky + dark, read_noise, ceiling, 500)  # this notebook's helper
field = obs.expose(exptime=t, n_reads=n, seed=1)                         # .image_e .image_clean .saturation_mask
field.write("m101_zwo_r_snr500.fits")                                    # SCI, SATMASK, CAT, CLEAN, COMP
```

The exposure times are what SNR 500 in a single 16.9 mas pixel costs; a requirement per square
arcsecond, per PSF, or on the integrated galaxy light is orders of magnitude cheaper. To explore
that, bin `field.image_e` (and `image_clean`) before measuring, or set `SNR_TARGET` lower and rerun.
Related worked examples: `../tutorial/06_fields_and_photometry.ipynb` (frames and truth catalogues),
`../../science_gallery/11_crowded_field_photometry.ipynb` (time series on a Gaia field).